# Multi-Agent Evidence-Graded Specialist RAG

Runs Architecture 1: Evidence-Graded Specialist Pipeline — a 5-agent sequential pipeline
that routes each question to the appropriate retrieval strategy, grades retrieved chunks
by evidential quality, generates an answer from the top-graded context, and verifies
the answer for unsupported claims before returning it.

**Pipeline:** Question → Agent 1 (Router) → Agent 2 (Adaptive Retrieval) → Agent 3 (Evidence Grader) → Agent 4 (Answer Generator) → Agent 5 (Hallucination Guard) → Verified Answer  
**Evaluation:** RAGAS and DeepEval metrics

In [8]:
import sys
sys.path.append("..")

import os
import re
import time
import json
import pandas as pd
from datetime import datetime
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams
import config
from ast import literal_eval
from ragas.metrics import NonLLMContextRecall, NonLLMContextPrecisionWithReference, BleuScore, RougeScore
from ragas import SingleTurnSample
from deepeval.evaluate import DisplayConfig, AsyncConfig
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import DeepEvalBaseLLM
from deepeval.metrics import (
    FaithfulnessMetric, ContextualRecallMetric,
    ContextualPrecisionMetric, AnswerRelevancyMetric, GEval
)
import deepeval
from huggingface_hub.utils import disable_progress_bars
disable_progress_bars()
pd.set_option('display.html.use_mathjax', False)
os.environ["DEEPEVAL_RETRY_MAX_ATTEMPTS"] = "2"

import logging
logging.basicConfig(level=logging.ERROR)

/tmp/ipykernel_29867/4217960929.py:18: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCaseParams
/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_29867/4217960929.py:21: DeprecationWarning: Importing NonLLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import NonLLMContextRecall
  from ragas.metrics import NonLLMContextRecall, NonLLMContextPrecisionWithReference

## Load Vector Store and Retriever Helpers

Reuses the same retriever construction functions as all prior experiments.

In [9]:
def load_chroma(embeddings, db_name=None, persist_dir=None):
    """Load an existing ChromaDB collection from disk."""
    from langchain_chroma import Chroma

    db_name = db_name or f"{config.DEFAULT_EMBEDDING}_pubmed_chromadb"
    persist_dir = persist_dir or str(config.VECTORSTORE_DIR / db_name)
    print(f"Loading ChromaDB from {persist_dir}")
    return Chroma(
        collection_name=db_name,
        persist_directory=persist_dir,
        embedding_function=embeddings,
    )


def get_cosine_retriever(vector_store, k=None):
    """Build a cosine similarity retriever."""
    k = k or config.TOP_K
    return vector_store.as_retriever(search_kwargs={"k": k})


def get_bm25_retriever(vector_store, k=None):
    """Build a BM25 sparse retriever from the full stored corpus."""
    k = k or config.TOP_K
    docs = [
        Document(page_content=x, metadata=m)
        for x, m in zip(vector_store.get()["documents"], vector_store.get()["metadatas"])
    ]
    retriever = BM25Retriever.from_documents(docs)
    retriever.k = k
    return retriever


def rrf(rank_lists, top_k=None, k=60):
    """Fuse multiple ranked retrieval lists using Reciprocal Rank Fusion."""
    top_k = top_k or config.TOP_K
    scores = defaultdict(lambda: {"doc": None, "score": 0.0})
    for docs in rank_lists:
        for rank, doc in enumerate(docs, 1):
            key = (doc.metadata["pubid"], doc.metadata["chunk_index"])
            if scores[key]["doc"] is None:
                scores[key]["doc"] = doc
            scores[key]["score"] += 1 / (k + rank)
    return [x["doc"] for x in sorted(scores.values(), key=lambda x: x["score"], reverse=True)[:top_k]]


def invoke_hybrid_retriever(query, dense_retriever, sparse_retriever, top_k=None):
    """Hybrid RRF retrieval: dense + BM25, fused by Reciprocal Rank Fusion."""
    top_k = top_k or config.TOP_K
    dense_docs = dense_retriever.invoke(query)
    sparse_docs = sparse_retriever.invoke(query)
    return rrf([dense_docs, sparse_docs], top_k=top_k)

In [10]:
def get_groq_llm(model=None, api_key=None):
    return ChatGroq(
        model=model or config.LLM_MODEL,
        api_key=api_key or config.GROQ_API_KEY,
    )


class GroqKeyRotator:
    """Cycles through config.GROQ_API_KEYS, rebuilding the LLM client on each rotation."""

    def __init__(self, model=None):
        if not config.GROQ_API_KEYS:
            raise ValueError("config.GROQ_API_KEYS is empty — set GROQ_API_KEY or GROQ_API_KEYS in .env")
        self.api_keys = config.GROQ_API_KEYS
        self.model = model or config.LLM_MODEL
        self.current_idx = 0
        print(f"Initialized GroqKeyRotator with {len(self.api_keys)} API key(s)")

    def get_llm(self):
        """Return a ChatGroq instance using the current API key."""
        return ChatGroq(
            model=self.model,
            api_key=self.api_keys[self.current_idx],
        )

    def rotate(self):
        """Advance to the next key, wrapping around."""
        self.current_idx = (self.current_idx + 1) % len(self.api_keys)
        print(f"Rotated to API key index {self.current_idx}")


RAG_PROMPT_TEMPLATE = """Use the following research contexts to answer the question.

Context:
{context}

Question: {question}

Answer based only on the provided context. Be precise and evidence-based.

Answer:"""

RAG_PROMPT = PromptTemplate(
    template=RAG_PROMPT_TEMPLATE,
    input_variables=["context", "question"],
)


def build_rag_chain(llm):
    return RAG_PROMPT | llm

## Agent 1: Router

Classifies the question into one of four types and selects the retrieval strategy.
- Conceptual questions → dense cosine retrieval (best for semantic similarity)
- Named-entity-heavy questions → BM25 (best for exact clinical terminology)
- Comparative and Multi-hop questions → hybrid RRF (maximises coverage across both arms)

In [6]:
ROUTER_PROMPT_TEMPLATE = """You are a biomedical question classifier.
Classify the given question into exactly one type and select the retrieval strategy.

Question types:
- Conceptual: asks about mechanisms, pathophysiology, or general relationships (e.g., "How does X cause Y?")
- Named-Entity: heavily involves specific drug names, gene variants, organisms, or identifiers (e.g., "Does BRCA1...")
- Comparative: compares two treatments, interventions, or conditions (e.g., "Does X outperform Y?")
- Multi-hop: requires combining evidence from multiple distinct studies or reasoning steps

Retrieval strategies:
- dense: cosine similarity embedding search (best for conceptual/semantic questions)
- bm25: keyword-based sparse retrieval (best for named entities and specific terminology)
- hybrid: combined dense + BM25 with rank fusion (best for comparative and multi-hop questions)

Respond with exactly this JSON format (no explanation, no markdown):
{{"question_type": "<Conceptual|Named-Entity|Comparative|Multi-hop>", "retrieval_strategy": "<dense|bm25|hybrid>"}}

Question: {question}

Classification:"""

ROUTER_PROMPT = PromptTemplate(
    template=ROUTER_PROMPT_TEMPLATE,
    input_variables=["question"],
)


def build_router_chain(llm):
    return ROUTER_PROMPT | llm


def parse_router_output(raw_output):
    """Parse JSON router output, with fallback to dense retrieval on parse failure."""
    text = raw_output.content if hasattr(raw_output, "content") else str(raw_output)
    text = text.strip()
    # Strip markdown fences if present
    text = re.sub(r"```(?:json)?\s*", "", text).strip().rstrip("`")
    try:
        result = json.loads(text)
        question_type = result.get("question_type", "Conceptual")
        strategy = result.get("retrieval_strategy", "dense")
        if strategy not in ("dense", "bm25", "hybrid"):
            strategy = "dense"
        return question_type, strategy
    except json.JSONDecodeError:
        # Fallback: try to extract strategy from raw text
        for strat in ("hybrid", "bm25", "dense"):
            if strat in text.lower():
                return "Conceptual", strat
        return "Conceptual", "dense"

## Agent 3: Evidence Grader (Rule-Based)

Scores each retrieved chunk using three heuristics without any LLM call:
1. **Section label priority**: RESULTS/CONCLUSIONS chunks score highest; BACKGROUND scores lowest.
2. **Study design keywords**: RCT, meta-analysis, and systematic review terms add score; case reports and pilot studies subtract.
3. **MeSH term overlap**: chunks whose MeSH terms match question keywords receive a bonus.

The top 5 chunks by combined score replace the raw retrieval ranking before generation.

In [7]:
# Section header priority scores (matched against structured PubMed section labels)
SECTION_PRIORITY = {
    "RESULTS": 3,
    "CONCLUSIONS": 3,
    "CONCLUSION": 3,
    "METHODS": 2,
    "METHOD": 2,
    "OBJECTIVE": 1,
    "AIM": 1,
    "BACKGROUND": 0,
    "INTRODUCTION": 0,
}

# Study design keyword scores applied to the full chunk text
STUDY_DESIGN_KEYWORDS = {
    "randomised controlled trial": 3,
    "randomized controlled trial": 3,
    "rct": 3,
    "meta-analysis": 3,
    "systematic review": 3,
    "cohort study": 2,
    "prospective study": 2,
    "double-blind": 2,
    "placebo-controlled": 2,
    "case-control": 1,
    "observational study": 1,
    "case report": -1,
    "case series": -1,
    "pilot study": -1,
    "animal model": -1,
    "in vitro": -1,
}


def _section_score(text):
    """Return the highest section priority score found in the chunk text."""
    text_upper = text.upper()
    best = 0
    for section, score in SECTION_PRIORITY.items():
        # Match section labels at the start of a line followed by a colon
        if re.search(rf"(?:^|\n){section}:\s", text_upper):
            best = max(best, score)
    return best


def _study_design_score(text):
    """Sum keyword scores from STUDY_DESIGN_KEYWORDS present in the chunk text."""
    text_lower = text.lower()
    total = 0
    for keyword, score in STUDY_DESIGN_KEYWORDS.items():
        if keyword in text_lower:
            total += score
    return total


def _mesh_overlap_score(metadata, question_lower):
    """Score a chunk based on the number of its MeSH terms that appear in the question."""
    raw = metadata.get("mesh_terms", "")
    if not raw:
        return 0
    try:
        mesh_list = json.loads(raw) if isinstance(raw, str) and raw.startswith("[") else [t.strip() for t in raw.split(",")]
    except Exception:
        mesh_list = [t.strip() for t in str(raw).split(",")]
    return sum(1 for m in mesh_list if m and m.lower() in question_lower)


def grade_chunk(doc, question_lower):
    """Compute a combined evidence grade for a single chunk.

    Args:
        doc: LangChain Document object.
        question_lower: Lowercase question string for MeSH overlap scoring.

    Returns:
        Float combined grade (higher is better).
    """
    section = _section_score(doc.page_content)
    design = _study_design_score(doc.page_content)
    mesh = _mesh_overlap_score(doc.metadata, question_lower)
    return float(section + design + mesh)


def select_top_chunks(docs, question, top_k=5):
    """Re-rank retrieved chunks by evidence grade and return the top-k.

    Args:
        docs: List of LangChain Document objects from the retrieval step.
        question: Original question string.
        top_k: Number of top-graded chunks to return.

    Returns:
        List of up to top_k Document objects sorted by descending evidence grade.
    """
    question_lower = question.lower()
    graded = [(doc, grade_chunk(doc, question_lower)) for doc in docs]
    graded.sort(key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in graded[:top_k]]

## Agent 5: Hallucination Guard

One LLM call after generation. The agent receives the generated answer and the five
retrieved chunks and identifies any factual claims not directly supported by the context.
Returns a verified answer with unsupported claims hedged or removed, plus a binary flag.

In [8]:
HALLUCINATION_GUARD_PROMPT_TEMPLATE = """You are a biomedical fact-checker.
You are given a generated answer and the research context chunks it was based on.
Your task is to identify any factual claims in the answer that are NOT directly supported by the context.

Context:
{context}

Generated Answer:
{answer}

Instructions:
1. List any claims in the answer that cannot be verified from the context above.
2. Produce a verified answer that removes or hedges unsupported claims.
   If all claims are supported, return the original answer unchanged.
3. Set hallucination_detected to true if any claims were removed or hedged.

Respond with exactly this JSON format (no explanation, no markdown):
{{"verified_answer": "<the verified answer text>", "hallucination_detected": <true|false>, "unsupported_claims": ["<claim 1>", "<claim 2>"]}}

Response:"""

HALLUCINATION_GUARD_PROMPT = PromptTemplate(
    template=HALLUCINATION_GUARD_PROMPT_TEMPLATE,
    input_variables=["context", "answer"],
)


def build_hallucination_guard_chain(llm):
    return HALLUCINATION_GUARD_PROMPT | llm


def parse_guard_output(raw_output, fallback_answer):
    """Parse JSON hallucination guard output with graceful fallback.

    Returns:
        Tuple of (verified_answer str, hallucination_detected bool, unsupported_claims list).
    """
    text = raw_output.content if hasattr(raw_output, "content") else str(raw_output)
    text = text.strip()
    text = re.sub(r"```(?:json)?\s*", "", text).strip().rstrip("`")
    try:
        result = json.loads(text)
        verified = result.get("verified_answer", fallback_answer)
        detected = bool(result.get("hallucination_detected", False))
        claims = result.get("unsupported_claims", [])
        return verified, detected, claims
    except json.JSONDecodeError:
        return fallback_answer, False, []

## Evaluation Functions

In [11]:
class GroqModel(DeepEvalBaseLLM):
    def __init__(self, model=None):
        self.model = model or get_groq_llm()

    def load_model(self):
        return self.model

    def generate(self, prompt: str) -> str:
        response = self.model.invoke(prompt)
        return response.content

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self):
        return "Groq Model"


def build_test_cases(eval_df):
    return [
        LLMTestCase(
            input=row["question"],
            actual_output=row["generated_answer"],
            retrieval_context=row["retrieved_contexts"],
            expected_output=row["golden_answer"],
        )
        for _, row in eval_df.iterrows()
    ]


def _make_ragas_scores_df(all_scores, metric_name):
    return pd.DataFrame({
        "question_index": range(len(all_scores)),
        metric_name: all_scores,
    })


def evaluate_ragas(eval_df, metric, results_file=None):
    """Evaluate with a non-LLM RAGAS metric over each row of the eval DataFrame."""
    metric_name = type(metric).__name__
    all_scores = []

    for _, row in eval_df.iterrows():
        sample = SingleTurnSample(
            user_input=row["question"],
            retrieved_contexts=list(row["retrieved_contexts"]),
            reference_contexts=row["golden_contexts"],
            reference=row["golden_answer"],
            response=row["generated_answer"]
        )
        score = metric.single_turn_score(sample)
        all_scores.append(score)

    avg = sum(all_scores) / len(all_scores)
    print(f"\n=== {metric_name}: {avg:.4f} (avg over {len(all_scores)} samples) ===")

    scores_df = _make_ragas_scores_df(all_scores, metric_name)
    if results_file:
        scores_df.to_csv(results_file, index=False)
        print(f"Saved scores to {results_file}")

    return all_scores, avg, scores_df


def build_ragas_combined(eval_df, score_dfs, results_file=None):
    combined = eval_df.copy().reset_index(drop=True)
    combined.insert(0, "question_index", range(len(combined)))
    combined = combined.rename(columns={"generated_answer": "generated_response"})
    for scores_df in score_dfs:
        combined = combined.merge(scores_df, on="question_index", how="left")
    if results_file:
        combined.to_csv(results_file, index=False)
        print(f"Saved combined RAGAS results to {results_file}")
    return combined


def _run_deepeval_slice(test_case_slice, api_key, model, metric_cls, threshold, delay, key_idx, metric_kwargs=None):
    llm = ChatGroq(model=model, api_key=api_key)
    results = []
    for i, test_case in enumerate(test_case_slice):
        try:
            metric = metric_cls(threshold=threshold, model=GroqModel(model=llm),
                                **metric_kwargs)
            result = deepeval.evaluate([test_case], metrics=[metric],
                                       display_config=DisplayConfig(
                                           verbose_mode=False,
                                           show_indicator=False,
                                           print_results=False),
                                       async_config=AsyncConfig(run_async=False))
            results.extend(result.test_results)
        except Exception as e:
            print(f"[Key {key_idx}] Error on case {i + 1}: {e}")
        if i < len(test_case_slice) - 1:
            time.sleep(delay)
    print(f"[Key {key_idx}] Done — {len(results)}/{len(test_case_slice)} cases evaluated")
    return results


def evaluate_deepeval_parallel(test_cases, metric_cls, key_rotator, threshold=0.5, results_file=None, delay=None, rows_per_key=None, metric_kwargs=None):
    rows_per_key = rows_per_key or config.PARALLEL_BUCKET_SIZE
    delay = delay or config.DEEPEVAL_DELAY_SECONDS
    api_keys = key_rotator.api_keys
    metric_kwargs = {} if metric_kwargs is None else dict(metric_kwargs)

    if not test_cases:
        raise ValueError("test_cases is empty.")

    total_capacity = len(api_keys) * rows_per_key
    if len(test_cases) > total_capacity:
        test_cases = test_cases[:total_capacity]

    slices = []
    for i, key in enumerate(api_keys):
        start = i * rows_per_key
        if start >= len(test_cases):
            break
        slices.append((key, i, test_cases[start: start + rows_per_key]))

    print(f"\n{len(test_cases)} cases split across {len(slices)} key(s):")
    ordered_results = [None] * len(slices)

    with ThreadPoolExecutor(max_workers=len(slices)) as executor:
        future_to_idx = {
            executor.submit(_run_deepeval_slice, s, key, key_rotator.model,
                            metric_cls, threshold, delay, i, metric_kwargs): idx
            for idx, (key, i, s) in enumerate(slices)
        }
        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            try:
                ordered_results[idx] = future.result()
            except Exception as e:
                print(f"[Key {idx}] Thread failed: {e}")
                ordered_results[idx] = []

    all_results = []
    res_df = None
    for key_results in ordered_results:
        all_results.extend(key_results)

    if all_results:
        scores = [r.metrics_data[0].score for r in all_results]
        metric_name = all_results[0].metrics_data[0].name
        average = sum(scores) / len(scores)
        print(f"\n=== {metric_name}: {average:.4f} (avg over {len(scores)} samples) ===")
        rows = []
        for r in all_results:
            rows.append({
                "question": r.input,
                "generated_answer": r.actual_output,
                "retrieved_contexts": r.retrieval_context,
                "golden_answer": r.expected_output,
                r.metrics_data[0].name: r.metrics_data[0].score,
            })
        res_df = pd.DataFrame(rows)
        if results_file:
            res_df.to_csv(results_file, index=False)
            print(f"Saved to {results_file}")

    return all_results, res_df

## Evidence-Graded RAG Pipeline

The `_run_slice` function implements the full 5-agent pipeline per row:
1. Agent 1 — Router: classifies question type and selects retrieval strategy (1 LLM call).
2. Agent 2 — Adaptive Retrieval: runs the selected retriever with k=8 to create a wider candidate pool.
3. Agent 3 — Evidence Grader: scores each chunk rule-based and keeps top 5 (no LLM call).
4. Agent 4 — Answer Generator: generates an answer from the top-graded context (1 LLM call).
5. Agent 5 — Hallucination Guard: verifies factual claims against the retrieved context (1 LLM call).

In [10]:
CANDIDATE_K = 8   # Wider candidate pool for the evidence grader to work from
GRADED_TOP_K = 5  # Final number of chunks passed to the generator


def _run_slice(retrievers, slice_df, api_key, model, delay, key_idx):
    """Process a contiguous slice of the evaluation set using the 5-agent pipeline.

    Args:
        retrievers: Dict with keys 'dense', 'bm25' mapping to retriever objects.
        slice_df: Contiguous DataFrame slice assigned to this key.
        api_key: Groq API key string dedicated to this thread.
        model: LLM model name.
        delay: Seconds to sleep between rows.
        key_idx: Key index used only for log prefixes.

    Returns:
        Copy of slice_df with new columns: route_type, route_strategy, retrieved_contexts,
        graded_contexts, generated_answer, hallucination_detected, unsupported_claims,
        and timing/token columns.
    """
    llm = ChatGroq(model=model, api_key=api_key)
    router_chain = build_router_chain(llm)
    rag_chain = build_rag_chain(llm)
    guard_chain = build_hallucination_guard_chain(llm)

    dense_retriever = retrievers["dense"]
    bm25_retriever = retrievers["bm25"]

    result_df = slice_df.copy().reset_index(drop=True)
    route_type_list = [None] * len(slice_df)
    route_strategy_list = [None] * len(slice_df)
    retrieved_contexts_list = [None] * len(slice_df)
    graded_contexts_list = [None] * len(slice_df)
    generated_answer_list = [None] * len(slice_df)
    hallucination_detected_list = [None] * len(slice_df)
    unsupported_claims_list = [None] * len(slice_df)
    total_time_list = [None] * len(slice_df)
    prompt_tokens_list = [None] * len(slice_df)
    completion_tokens_list = [None] * len(slice_df)
    total_tokens_list = [None] * len(slice_df)

    for row_idx, (_, row) in enumerate(slice_df.iterrows()):
        question = row["question"]
        try:
            time_start = time.perf_counter()

            # Agent 1: Router
            router_result = router_chain.invoke({"question": question})
            question_type, strategy = parse_router_output(router_result)

            # Agent 2: Adaptive Retrieval (k=8 candidate pool)
            if strategy == "bm25":
                bm25_retriever.k = CANDIDATE_K
                candidate_docs = bm25_retriever.invoke(question)
            elif strategy == "hybrid":
                candidate_docs = invoke_hybrid_retriever(
                    question, dense_retriever, bm25_retriever, top_k=CANDIDATE_K
                )
            else:  # dense
                candidate_docs = dense_retriever.invoke(question)

            # Agent 3: Evidence Grader (rule-based, no LLM call)
            graded_docs = select_top_chunks(candidate_docs, question, top_k=GRADED_TOP_K)

            # Agent 4: Answer Generator
            gen_result = rag_chain.invoke({"context": graded_docs, "question": question})
            draft_answer = gen_result.content

            # Token usage from generation call
            token_usage = gen_result.response_metadata.get("token_usage", {})

            # Agent 5: Hallucination Guard
            context_text = "\n\n".join(doc.page_content for doc in graded_docs)
            guard_result = guard_chain.invoke({"context": context_text, "answer": draft_answer})
            verified_answer, hal_detected, bad_claims = parse_guard_output(guard_result, draft_answer)

            total_time = time.perf_counter() - time_start

            route_type_list[row_idx] = question_type
            route_strategy_list[row_idx] = strategy
            retrieved_contexts_list[row_idx] = [doc.page_content for doc in candidate_docs]
            graded_contexts_list[row_idx] = [doc.page_content for doc in graded_docs]
            generated_answer_list[row_idx] = verified_answer
            hallucination_detected_list[row_idx] = hal_detected
            unsupported_claims_list[row_idx] = bad_claims
            total_time_list[row_idx] = total_time
            prompt_tokens_list[row_idx] = token_usage.get("prompt_tokens", 0)
            completion_tokens_list[row_idx] = token_usage.get("completion_tokens", 0)
            total_tokens_list[row_idx] = token_usage.get("total_tokens", 0)

        except Exception as e:
            print(f"[Key {key_idx}] Error on '{question[:50]}...': {e}")

        if row_idx < len(slice_df) - 1:
            time.sleep(delay)

    result_df["route_type"] = route_type_list
    result_df["route_strategy"] = route_strategy_list
    result_df["retrieved_contexts"] = retrieved_contexts_list
    result_df["graded_contexts"] = graded_contexts_list
    result_df["generated_answer"] = generated_answer_list
    result_df["hallucination_detected"] = hallucination_detected_list
    result_df["unsupported_claims"] = unsupported_claims_list
    result_df["total_time"] = total_time_list
    result_df["prompt_tokens"] = prompt_tokens_list
    result_df["completion_tokens"] = completion_tokens_list
    result_df["total_tokens"] = total_tokens_list

    completed = sum(1 for x in generated_answer_list if x is not None)
    print(f"[Key {key_idx}] Done — {completed}/{len(slice_df)} rows collected")
    return result_df


def run_evidence_rag_parallel(retrievers, df, key_rotator, rows_per_key=None, delay=None):
    """Assign a contiguous slice of rows to each API key and run all slices in parallel.

    Args:
        retrievers: Dict with keys 'dense' and 'bm25' mapping to retriever objects.
        df: DataFrame with at least question and golden_answer columns.
        key_rotator: GroqKeyRotator providing API keys and LLM model name.
        rows_per_key: Max rows assigned to each key (default config.PARALLEL_BUCKET_SIZE).
        delay: Seconds between rows (default config.PARALLEL_DELAY_SECONDS).

    Returns:
        A copy of df with new columns added by _run_slice.
    """
    rows_per_key = rows_per_key or config.PARALLEL_BUCKET_SIZE
    delay = delay or config.PARALLEL_DELAY_SECONDS
    api_keys = key_rotator.api_keys

    if len(df) == 0:
        raise ValueError("DataFrame is empty — nothing to evaluate.")

    total_capacity = len(api_keys) * rows_per_key
    if len(df) > total_capacity:
        print(f"Warning: {len(df)} rows exceed capacity ({total_capacity}). Truncating.")
        df = df.iloc[:total_capacity]

    slices = []
    for i, key in enumerate(api_keys):
        start = i * rows_per_key
        if start >= len(df):
            break
        slices.append((key, i, df.iloc[start: start + rows_per_key]))

    print(f"\n{len(df)} rows split across {len(slices)} key(s) ({rows_per_key} rows/key max):")
    for key, i, s in slices:
        print(f"  Key {i}: rows {i * rows_per_key}–{i * rows_per_key + len(s) - 1} ({len(s)} rows)")
    print()

    ordered_results = [None] * len(slices)

    with ThreadPoolExecutor(max_workers=len(slices)) as executor:
        future_to_idx = {
            executor.submit(
                _run_slice,
                retrievers, s, key, key_rotator.model, delay, i,
            ): idx
            for idx, (key, i, s) in enumerate(slices)
        }
        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            _, _, s = slices[idx]
            try:
                ordered_results[idx] = future.result()
            except Exception as e:
                print(f"[Key {idx}] Thread failed: {e}")
                fallback_df = s.copy().reset_index(drop=True)
                for col in ["route_type", "route_strategy", "retrieved_contexts",
                             "graded_contexts", "generated_answer",
                             "hallucination_detected", "unsupported_claims"]:
                    fallback_df[col] = [None] * len(s)
                ordered_results[idx] = fallback_df

    final_df = pd.concat(ordered_results, ignore_index=True)
    completed = final_df["generated_answer"].notna().sum()
    print(f"\nCompleted {completed}/{len(df)} questions total")
    print(f"\nAverage Time Per Query: {final_df['total_time'].mean():.4f}s")
    print(f"\nAverage Total Tokens Per Query: {final_df['total_tokens'].mean():.1f}")
    hal_count = final_df["hallucination_detected"].sum()
    print(f"\nHallucinations detected: {hal_count}/{len(df)} questions ({100*hal_count/len(df):.1f}%)")
    if "route_strategy" in final_df.columns:
        print(f"\nRouting distribution:\n{final_df['route_strategy'].value_counts().to_string()}")
    return final_df

---
## Setup

In [ ]:
embedding_key = config.DEFAULT_EMBEDDING
embeddings = HuggingFaceEmbeddings(model_name=config.EMBEDDING_MODELS[embedding_key])
key_rotator = GroqKeyRotator()
llm = key_rotator.get_llm()
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"Embedding: {config.EMBEDDING_MODELS[embedding_key]}")
print(f"LLM: {key_rotator.model}")
print(f"Run timestamp: {timestamp}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Initialized GroqKeyRotator with 10 API key(s)
Embedding: sentence-transformers/all-MiniLM-L6-v2
LLM: llama-3.3-70b-versatile
Run timestamp: 20260511_140805


In [12]:
#### TO DELETE ####

embedding_key = config.DEFAULT_EMBEDDING
embeddings = HuggingFaceEmbeddings(model_name=config.EMBEDDING_MODELS[embedding_key])
key_rotator = GroqKeyRotator()
llm = key_rotator.get_llm()
timestamp = "20260511_140805"
print(f"Embedding: {config.EMBEDDING_MODELS[embedding_key]}")
print(f"LLM: {key_rotator.model}")
print(f"Run timestamp: {timestamp}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Initialized GroqKeyRotator with 10 API key(s)
Embedding: sentence-transformers/all-MiniLM-L6-v2
LLM: llama-3.3-70b-versatile
Run timestamp: 20260511_140805


## Prepare Evaluation Sample

In [13]:
golden_df = pd.read_csv(config.DATA_PROCESSED_DIR / "golden_dataset_complete.csv")
print(f"Loaded {len(golden_df)} samples from golden_dataset_complete.csv")
golden_df['golden_contexts'] = golden_df['golden_contexts'].apply(literal_eval)
golden_df.info()

Loaded 200 samples from golden_dataset_complete.csv
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   question_idx     200 non-null    int64 
 1   question         200 non-null    object
 2   golden_answer    200 non-null    object
 3   golden_contexts  200 non-null    object
 4   query_type       200 non-null    object
 5   pubids_needed    200 non-null    object
dtypes: int64(1), object(5)
memory usage: 9.5+ KB


## Load Vector Store and Build Retrievers

In [13]:
vector_store = load_chroma(embeddings, db_name=f"{embedding_key}_pubmed_chromadb")

# Dense retriever with k=8 (wider candidate pool for evidence grader)
dense_retriever = get_cosine_retriever(vector_store, k=CANDIDATE_K)
bm25_retriever = get_bm25_retriever(vector_store, k=CANDIDATE_K)

retrievers = {"dense": dense_retriever, "bm25": bm25_retriever}

Loading ChromaDB from /content/vectorstores/minilm_pubmed_chromadb


## Smoke Test: Single Question

In [ ]:
smoke_result = run_evidence_rag_parallel(retrievers, golden_df.head(1), key_rotator)
smoke_result


1 rows split across 1 key(s) (20 rows/key max):
  Key 0: rows 0–0 (1 rows)

[Key 0] Done — 1/1 rows collected

Completed 1/1 questions total

Average Time Per Query: 1.6668s

Average Total Tokens Per Query: 1835.0

Hallucinations detected: 0/1 questions (0.0%)

Routing distribution:
route_strategy
dense    1


,question_idx,question,golden_answer,golden_contexts,query_type,pubids_needed,route_type,route_strategy,retrieved_contexts,graded_contexts,generated_answer,hallucination_detected,unsupported_claims,total_time,prompt_tokens,completion_tokens,total_tokens
0,0,Is there a relationship between rheumatoid art...,Based on data derived from self-reported healt...,"[1,412 individuals attending the University of...",Single-hop,['10783841'],Conceptual,dense,[CONCLUSIONS: Based on data derived from self-...,[CONCLUSIONS: Based on data derived from self-...,"Yes, there is a relationship between rheumatoi...",False,[],1.666845,1665,170,1835


## Run Evidence-Graded RAG on Full Evaluation Set

In [ ]:
eval_dataset = run_evidence_rag_parallel(retrievers, golden_df, key_rotator)
eval_dataset.to_csv(
    str(config.RESULTS_EVALSETS_DIR / f"evidence_graded_rag_{embedding_key}_chroma_{timestamp}.csv"),
    index=False
)
print(f"Generated {len(eval_dataset)} answers")


200 rows split across 10 key(s) (20 rows/key max):
  Key 0: rows 0–19 (20 rows)
  Key 1: rows 20–39 (20 rows)
  Key 2: rows 40–59 (20 rows)
  Key 3: rows 60–79 (20 rows)
  Key 4: rows 80–99 (20 rows)
  Key 5: rows 100–119 (20 rows)
  Key 6: rows 120–139 (20 rows)
  Key 7: rows 140–159 (20 rows)
  Key 8: rows 160–179 (20 rows)
  Key 9: rows 180–199 (20 rows)

[Key 8] Error on 'How do central venous pressure and colloid preload...': Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01hzs3gv6vfsf9qg002zv18ps6` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 11173, Requested 4380. Please try again in 17.764999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
[Key 2] Done — 20/20 rows collected
[Key 4] Done — 20/20 rows collected
[Key 1] Done — 20/20 rows collected
[Key 5] Done — 20/20 rows collected
[Key 

In [ ]:
error_dataset = eval_dataset[eval_dataset["generated_answer"].isna()]
print(f"Number of errored rows: {len(error_dataset)}")
subset_golden_df = golden_df[golden_df["question"].isin(error_dataset["question"])]
subset_eval_df = run_evidence_rag_parallel(retrievers, subset_golden_df, key_rotator, rows_per_key=1, delay=4)
subset_eval_df

Number of errored rows: 1

1 rows split across 1 key(s) (1 rows/key max):
  Key 0: rows 0–0 (1 rows)

[Key 0] Done — 1/1 rows collected

Completed 1/1 questions total

Average Time Per Query: 2.2353s

Average Total Tokens Per Query: 1881.0

Hallucinations detected: 1/1 questions (100.0%)

Routing distribution:
route_strategy
hybrid    1


,question_idx,question,golden_answer,golden_contexts,query_type,pubids_needed,route_type,route_strategy,retrieved_contexts,graded_contexts,generated_answer,hallucination_detected,unsupported_claims,total_time,prompt_tokens,completion_tokens,total_tokens
0,175,How do central venous pressure and colloid pre...,Central venous pressure alone does not reliabl...,[A meta-analysis of 43 studies found that base...,Multi-Hop,"['23774337', '15962678']",Comparative,hybrid,"[OBJECTIVE: This prospective, randomized, doub...",[BACKGROUND: Despite a previous meta-analysis ...,"According to the provided context, central ven...",True,[colloid preloading may be more effective in c...,2.235292,1667,214,1881


In [ ]:
for idx, row in subset_eval_df.iterrows():
    eval_dataset.at[row["question_idx"], "generated_answer"] = row["generated_answer"]
    eval_dataset.at[row["question_idx"], "route_type"] = row["route_type"]
    eval_dataset.at[row["question_idx"], "route_strategy"] = row["route_strategy"]
    eval_dataset.at[row["question_idx"], "retrieved_contexts"] = row["retrieved_contexts"]
    eval_dataset.at[row["question_idx"], "graded_contexts"] = row["graded_contexts"]
    eval_dataset.at[row["question_idx"], "hallucination_detected"] = row["hallucination_detected"]
    eval_dataset.at[row["question_idx"], "unsupported_claims"] = row["unsupported_claims"]
    eval_dataset.at[row["question_idx"], "total_time"] = row["total_time"]
    eval_dataset.at[row["question_idx"], "prompt_tokens"] = row["prompt_tokens"]
    eval_dataset.at[row["question_idx"], "completion_tokens"] = row["completion_tokens"]
    eval_dataset.at[row["question_idx"], "total_tokens"] = row["total_tokens"]

len(eval_dataset[eval_dataset["generated_answer"].isna()])

0

In [ ]:
eval_dataset.to_csv(str(config.RESULTS_EVALSETS_DIR / f"evidence_graded_rag_{embedding_key}_chroma_{timestamp}.csv"))

### RAGAS Evaluation

In [ ]:
ragas_cr_scores, ragas_cr_avg, ragas_cr_df = evaluate_ragas(
    eval_dataset, NonLLMContextRecall(),
    results_file=str(config.RESULTS_RAGAS_DIR / f"evidence_graded_rag_{embedding_key}_context_recall_{timestamp}.csv")
)


=== NonLLMContextRecall: 0.2121 (avg over 200 samples) ===
Saved scores to /content/results/ragas/evidence_graded_rag_minilm_context_recall_20260511_140805.csv


In [ ]:
ragas_cp_scores, ragas_cp_avg, ragas_cp_df = evaluate_ragas(
    eval_dataset, NonLLMContextPrecisionWithReference(),
    results_file=str(config.RESULTS_RAGAS_DIR / f"evidence_graded_rag_{embedding_key}_context_precision_{timestamp}.csv")
)


=== NonLLMContextPrecisionWithReference: 0.2959 (avg over 200 samples) ===
Saved scores to /content/results/ragas/evidence_graded_rag_minilm_context_precision_20260511_140805.csv


In [ ]:
ragas_bleu_scores, ragas_bleu_avg, ragas_bleu_df = evaluate_ragas(
    eval_dataset, BleuScore(),
    results_file=str(config.RESULTS_RAGAS_DIR / f"evidence_graded_rag_{embedding_key}_bleu_{timestamp}.csv")
)


=== BleuScore: 0.2269 (avg over 200 samples) ===
Saved scores to /content/results/ragas/evidence_graded_rag_minilm_bleu_20260511_140805.csv


In [ ]:
ragas_rouge_scores, ragas_rouge_avg, ragas_rouge_df = evaluate_ragas(
    eval_dataset, RougeScore(rouge_type="rougeL", mode="fmeasure"),
    results_file=str(config.RESULTS_RAGAS_DIR / f"evidence_graded_rag_{embedding_key}_rouge_{timestamp}.csv")
)


=== RougeScore: 0.3402 (avg over 200 samples) ===
Saved scores to /content/results/ragas/evidence_graded_rag_minilm_rouge_20260511_140805.csv


In [ ]:
combined_ragas = build_ragas_combined(
    eval_dataset,
    [ragas_cr_df, ragas_cp_df, ragas_bleu_df, ragas_rouge_df],
    results_file=str(config.RESULTS_RAGAS_DIR / f"evidence_graded_rag_{embedding_key}_combined_{timestamp}.csv")
)

Saved combined RAGAS results to /content/results/ragas/evidence_graded_rag_minilm_combined_20260511_140805.csv


### DeepEval Evaluation

In [14]:
### TO DELETE ###
eval_dataset = pd.read_csv(str(config.RESULTS_EVALSETS_DIR / f"evidence_graded_rag_{embedding_key}_chroma_{timestamp}.csv"))
eval_dataset['retrieved_contexts'] = eval_dataset['retrieved_contexts'].apply(literal_eval)
eval_dataset['graded_contexts'] = eval_dataset['graded_contexts'].apply(literal_eval)
eval_dataset['golden_contexts'] = eval_dataset['golden_contexts'].apply(literal_eval)
eval_dataset['unsupported_claims'] = eval_dataset['unsupported_claims'].apply(literal_eval)
eval_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Unnamed: 0              200 non-null    int64  
 1   question_idx            200 non-null    int64  
 2   question                200 non-null    object 
 3   golden_answer           200 non-null    object 
 4   golden_contexts         200 non-null    object 
 5   query_type              200 non-null    object 
 6   pubids_needed           200 non-null    object 
 7   route_type              200 non-null    object 
 8   route_strategy          200 non-null    object 
 9   retrieved_contexts      200 non-null    object 
 10  graded_contexts         200 non-null    object 
 11  generated_answer        200 non-null    object 
 12  hallucination_detected  200 non-null    bool   
 13  unsupported_claims      200 non-null    object 
 14  total_time              200 non-null    fl

In [15]:
test_cases = build_test_cases(eval_dataset)
de_key_rotator = GroqKeyRotator(model="openai/gpt-oss-120b")

Initialized GroqKeyRotator with 10 API key(s)


In [ ]:
deepeval_cr, deepeval_cr_df = evaluate_deepeval_parallel(
    test_cases, ContextualRecallMetric, de_key_rotator,
    results_file=str(config.RESULTS_DEEPEVAL_DIR / f"evidence_graded_rag_{embedding_key}_ctx_recall_{timestamp}.csv")
)

In [ ]:
deepeval_cp, deepeval_cp_df = evaluate_deepeval_parallel(
    test_cases, ContextualPrecisionMetric, de_key_rotator,
    results_file=str(config.RESULTS_DEEPEVAL_DIR / f"evidence_graded_rag_{embedding_key}_ctx_precision_{timestamp}.csv")
)

In [17]:
deepeval_f, deepeval_f_df = evaluate_deepeval_parallel(
    test_cases, FaithfulnessMetric, de_key_rotator,
    results_file=str(config.RESULTS_DEEPEVAL_DIR / f"evidence_graded_rag_{embedding_key}_faithfulness_{timestamp}.csv")
)


200 cases split across 10 key(s):


⚠ WARNING: No hyperparameters logged.
» ]8;id=532666;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 5.57s | token cost: None)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Warning: Could not update test run on disk: [Errno 2] No such file or directory: 
'.deepeval/.temp_test_run_data.json'

Warning: Could not load test run from disk: [Errno 2] No such file or directory: 
'.deepeval/.temp_test_run_data.json'

...

Warning: Could not update test run on disk: [Errno 2] No such file or directory: 
'.deepeval/.temp_test_run_data.json'

Warning: Could not load test run from disk: [Errno 2] No such file or directory: 
'.deepeval/.temp_test_run_data.json'

⚠ WARNING: No hyperparameters logged.
» ]8;id=592836;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 23.87s | token cost: None)
» Test Results (2 total tests):
   » Pass Rate: 100.0% | Passed: 2 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

[Key 8] Done — 17/20 cases evaluated

=== Faithfulness: 0.9674 (avg over 187 samples) ===
Saved to /content/results/deepeval/evidence_graded_rag_minilm_faithfulness_20260511_140805.csv


In [16]:
def resume_deepeval_from_csv(eval_dataset, existing_results_file, metric_cls, key_rotator,
    metric_column, threshold=0.5, delay=None, rows_per_key=None, metric_kwargs=None):
    """Resume an interrupted DeepEval run from an existing CSV.

    Identifies missing or incomplete questions in an existing DeepEval results
    file, recomputes only those rows, merges the new scores, and overwrites the original file.

    Matching is performed using the question text rather than row position,
    making the method robust to out-of-order, partially completed, or shuffled CSV files.

    Args:
        eval_dataset: Full evaluation DataFrame.
        existing_results_file: Existing DeepEval CSV path.
        metric_cls: DeepEval metric class.
        key_rotator: API key rotator.
        metric_column: Metric column name in CSV.
        threshold: DeepEval threshold.
        delay: Delay between API calls.
        rows_per_key: Rows per API key.
        metric_kwargs: Optional metric init kwargs.

    Returns:
        Final merged DataFrame.
    """
    existing_df = pd.read_csv(existing_results_file)

    # Questions already successfully evaluated
    completed_questions = set(existing_df.loc[
        existing_df[metric_column].notna(),"question"].astype(str))

    # Missing/incomplete rows anywhere in dataset
    missing_df = eval_dataset[~eval_dataset["question"].astype(str)
                              .isin(completed_questions)].copy()
    if missing_df.empty:
        print("All rows already completed.")
        return existing_df
    print(f"Need to recompute {len(missing_df)} rows.")

    # Build test cases only for missing rows
    test_cases = build_test_cases(missing_df)

    # Run DeepEval only for missing rows
    _, new_results_df = evaluate_deepeval_parallel(test_cases, metric_cls,
        key_rotator, threshold=threshold, delay=delay, rows_per_key=rows_per_key,
        metric_kwargs=metric_kwargs)

    # Remove old incomplete duplicates
    existing_df = existing_df[~existing_df["question"].astype(str).isin(
            new_results_df["question"].astype(str))]

    # Merge
    final_df = pd.concat([existing_df, new_results_df], ignore_index=True)
    if "question_idx" in final_df.columns:
      final_df = final_df.drop(columns=["question_idx"])
    final_df = final_df.merge(eval_dataset[["question", "question_idx"]], on="question", how="left")
    cols = ["question_idx"] + [c for c in final_df.columns if c != "question_idx"]
    final_df = final_df.sort_values("question_idx").reset_index(drop=True)

    final_df.to_csv(existing_results_file, index=False)
    print(f"Completed: {len(final_df)}/{len(eval_dataset)} rows")

    return final_df

In [17]:
faithfulness_df = resume_deepeval_from_csv(
    eval_dataset, str(config.RESULTS_DEEPEVAL_DIR / f"evidence_graded_rag_{embedding_key}_faithfulness_{timestamp}.csv"), FaithfulnessMetric,
    de_key_rotator, "Faithfulness", delay=40, rows_per_key=3
)

Need to recompute 13 rows.

13 cases split across 5 key(s):


⚠ WARNING: No hyperparameters logged.
» ]8;id=121628;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 7.44s | token cost: None)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

[Key 4] Done — 1/1 cases evaluated


Warning: Could not update test run on disk: [Errno 2] No such file or directory: 
'.deepeval/.temp_test_run_data.json'

...

✓ Evaluation completed 🎉! (time taken: 13.76s | token cost: None)
» Test Results (3 total tests):
   » Pass Rate: 100.0% | Passed: 3 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

[Key 0] Done — 3/3 cases evaluated


⚠ WARNING: No hyperparameters logged.
» ]8;id=625446;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 16.14s | token cost: None)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

[Key 3] Done — 3/3 cases evaluated

=== Faithfulness: 0.9744 (avg over 13 samples) ===
Completed: 200/200 rows


In [ ]:
# Defining Answer Correctness
evaluation_steps = [
    "Compare the generated answer with the reference answer in the context of the original biomedical question.",
    "Check whether the generated answer contains factually correct biomedical information and no contradictions to the reference answer.",
    "Verify that all clinically important facts needed to answer the question are present and no critical information is missing.",
    "Ignore wording differences, but penalize incorrect medical claims, unsupported conclusions, or misleading clinical interpretations."
]
deepeval_ac = evaluate_deepeval_parallel(
    test_cases, GEval, de_key_rotator,
    results_file=str(config.RESULTS_DEEPEVAL_DIR / f"evidence_graded_rag_{embedding_key}_answer_correctness_{timestamp}.csv"),
    metric_kwargs={
        "name": "AnswerCorrectness",
        "evaluation_steps": evaluation_steps,
        "evaluation_params": [
            LLMTestCaseParams.INPUT,
            LLMTestCaseParams.ACTUAL_OUTPUT,
            LLMTestCaseParams.EXPECTED_OUTPUT,
        ]}
)


200 cases split across 10 key(s):
[Key 0] Error on case 1: 'NoneType' object has no attribute 'save'


⚠ WARNING: No hyperparameters logged.
» ]8;id=98649;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 0.96s | token cost: None)
» Test Results (3 total tests):
   » Pass Rate: 66.67% | Passed: 2 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Warning: Could not update test run on disk: [Errno 2] No such file or directory: 
'.deepeval/.temp_test_run_data.json'

Warning: Could not load test run from disk: [Errno 2] No such file or directory: 
'.deepeval/.temp_test_run_data.json'

...

Warning: Could not update test run on disk: [Errno 2] No such file or directory: 
'.deepeval/.temp_test_run_data.json'

Warning: Could not load test run from disk: [Errno 2] No such file or directory: 
'.deepeval/.temp_test_run_data.json'

⚠ WARNING: No hyperparameters logged.
» ]8;id=5119;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 1.33s | token cost: None)
» Test Results (5 total tests):
   » Pass Rate: 60.0% | Passed: 3 | Failed: 2

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

[Key 7] Done — 20/20 cases evaluated

=== AnswerCorrectness [GEval]: 0.7095 (avg over 199 samples) ===
Saved to /content/results/deepeval/evidence_graded_rag_minilm_answer_correctness_20260511_140805.csv
